# 実習② 夏目漱石『こころ』の分析

In [ ]:
# ライブラリの読み込み
library('tidyverse')
library('RMeCab')

In [ ]:
dict = 'dict/kokoro_utf8.dic'

In [ ]:
getwd()

In [ ]:
kokoro = read.delim('data/textmining/kokoro.tsv', header=T, sep='\t', stringsAsFactor=F, fileEncoding='utf8')

In [ ]:
kokoro[1:2,] 

In [ ]:
# 段落の長さの分布
kokoro[, 'content'] %>% str_length() %>% hist(breaks=40, xlab='Paragraph length', main='Histogram of paragraph length')

In [ ]:
kokoro['length'] = kokoro[, 'content'] %>% str_length()

In [ ]:
boxplot(length ~ part_id, data=kokoro, main='Paragraph length of each part')

In [ ]:
# 全体を通すセクション用idを作る
kokoro['section_id2'] = kokoro['part_id'] * 100 + kokoro['section_id']

In [ ]:
boxplot(length ~ section_id2, data=kokoro, main='Paragraph length of each section')

In [ ]:
# 分析のため各部ごとに文章を結合する
parts = kokoro %>% 
    group_by(part_id) %>% 
    summarise(text = paste0(content, collapse=''))
parts = as.data.frame(parts)

In [ ]:
dim(parts)

In [ ]:
colnames(parts)

In [ ]:
parts[, 'text'] %>% str_length() 

In [ ]:
count_noun = docMatrixDF(parts[,'text'], pos=c('名詞'), , dic=dict)

In [ ]:
count_noun %>% head()

In [ ]:
# 全体を集計する
freq_noun = count_noun %>% rowSums()

In [ ]:
# 全体を集計する
freq_noun %>% sort(decreasing=T) %>% plot(main='Distribution of noun frequency', xlab='Rank', ylab='Frequency')

In [ ]:
freq_noun %>% sort(decreasing=T) %>% plot(main='Distribution of noun frequency', xlab='Rank (log)', ylab='Frequency (log)', log='xy')

In [ ]:
# Web環境のみ
library(showtext)
font_add_google("Noto Sans JP", "jpfont")  # Googleフォントから
showtext_auto()

In [ ]:
freq_noun %>% 
    sort() %>% 
    tail(30) %>%
    barplot(horiz=T, las=1, main='Top 30 nouns', xlab='Frequency', cex.names =0.8)

In [ ]:
# ストップワードを設定する
stopwords = c('事','の','よう','それ','もの', '人', '何','一', 'ん','方','二','前','気','中','上','今','ため')

In [ ]:
freq_noun[!names(freq_noun) %in% stopwords] %>%
    sort() %>% 
    tail(30) %>% 
    barplot(horiz=T, las=1, main='Top 30 nouns', xlab='Frequency', cex.names =0.6)

In [ ]:
# ストップワードをさらに増やす
stopwords = c('事','の','よう','それ','もの', '人', '何','一', 'ん','方','二','前','気','中','上','今','ため', '時', 'そこ', 'どこ', 'これ', 'そう')

freq_noun[!names(freq_noun) %in% stopwords] %>% 
    sort() %>% 
    tail(30) %>% 
    barplot(horiz=T, las=1, main='Top 30 nouns', xlab='Frequency', cex.names =0.6)

In [ ]:
# PCA Principal Component Analysis
# 主成分分析とバイプロットを使って、各部でどの単語が多いかを図示する
mat = count_noun 

# 全体での頻度が多いもの(n > 50)を選び出す
mat = mat[rowSums(mat) > 50, ]

# ストップワードを除く
mat = mat[!row.names(mat) %in% stopwords, ]

colnames(mat) = c('第一部', '第二部', '第三部')
mat_t = t(mat)

In [ ]:
mat %>% head()

In [ ]:
mat_t %>% head()

In [ ]:
# 出現確率(割合)に変換する
# 単語の頻度 / 部の長さ
(mat_t / colSums(mat)) 

In [ ]:
# 割合データを主成分分析にかける
result = (mat_t / colSums(mat)) %>% prcomp()

In [ ]:
# 単語の頻度
biplot(result)
## べき分布なので、頻度の多い単語に引っ張られて他のがよくわからない

In [ ]:
ratio = mat_t / colSums(mat)
ratio_t = t(ratio)

In [ ]:
ratio

In [ ]:
# 出現確率の比をとる
# 各部の出現確率 / (第一部の出現確率＋第二部の出現確率＋第三部の出現確率)
(ratio_t / colSums(ratio))

In [ ]:
result = (ratio_t / colSums(ratio))  %>% prcomp()

In [ ]:
biplot(result)

In [ ]:
# 名詞のデータセット
count_noun = docMatrixDF(parts[,'text'], pos=c('名詞'), dic=dict)

In [ ]:
count_noun

In [ ]:
# 共起分析
# バイグラムを使った分析

In [ ]:
bigram = docDF(parts, col='text', type=1, pos=c('名詞'), N=2, nDF=1, dic=dict)

In [ ]:
bigram %>% head()

In [ ]:
# distribution
bigram['freq'] = bigram[,5:7] %>% rowSums() 

In [ ]:
bigram[,'freq'] %>% sort(decreasing=T) %>% plot()

In [ ]:
bigram[,'freq'] %>% sort(decreasing=T) %>% plot(log='xy')

In [ ]:
# bigram = docDF(parts, col='text', type=1, pos=c('名詞'), N=2, nDF=1, dic=kokoro_dict) # N=2でバイグラムを指定I

In [ ]:
# igraph: ネットワーク分析用のパッケージ
# install.packages('igraph')
library('igraph')

In [ ]:
bigram %>% head()

In [ ]:
net = bigram %>% 
    select(N1, N2, freq) %>%  
    filter(freq > 10) %>% # 頻度が10以上
    filter(! N1 %in% stopwords) %>% # n1, n2のいずれからもストップワードを除去
    filter(! N2 %in% stopwords)


In [ ]:
options(repr.plot.width=16, repr.plot.height=9)

In [ ]:
font_family = 'jpfont'
g = net %>% graph_from_data_frame() # %>% tkplot(vertex.color='SkyBlue', vertex.size=22)
layout = layout_with_fr(g)
plot(g, layout=layout, vertex.label.family = font_family, vertext.size=22, vertex.color='SkyBlue', niter=500)

# R Studioの場合はtkplotの方が使いやすい
# net %>% graph_from_data_frame() %>% tkplot(vertex.color='SkyBlue', vertex.size=22)

In [ ]:
# 関数を定義
graph_plot = function(net){
    g = net %>% graph_from_data_frame() # %>% tkplot(vertex.color='SkyBlue', vertex.size=22)
    layout = layout_with_fr(g, niter=500)
    plot(g, layout=layout, vertex.label.family = font_family, vertext.size=22, vertex.color='SkyBlue')
    }

In [ ]:
net = bigram %>% 
    select(N1, N2, freq) %>%  
    filter(freq > 10) %>% # 頻度が10以上
    filter(! N1 %in% stopwords) %>% # n1, n2のいずれからもストップワードを除去
    filter(! N2 %in% stopwords)

In [ ]:
net %>% graph_plot()

In [ ]:
net = bigram %>% 
    select(N1, N2, freq=Row3) %>% # 第三部
    filter(freq > 10)  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords)

net %>% graph_plot()

In [ ]:
## 登場人物と単語の結びつき

# 品詞の種類を変えてみる
bigram = docDF(parts,col='text', type=1, pos=c('名詞', '動詞', '形容詞', '副詞'), N=2, nDF=1, dic=dict)

In [ ]:
# 「お嬢さん」
net = bigram %>% 
    select(N1, N2, freq=Row3) %>% 
    filter(freq > 1)  %>% 
    filter(N1 == 'お嬢さん' | N2 == 'お嬢さん')  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords)

net %>% graph_plot()

In [ ]:
stopwords = c('事','の','よう','それ','もの', '人', '何','一', 'ん','方','二','前','気','中','上','今','ため', '時', 'そこ', 'どこ', 'これ', 'そう',
               'いる', 'なる', 'する', 'いう', 'ある')
 

In [ ]:
# 「私」

# 抽出条件を色々試してみる
bigram %>% 
    filter(N1 == '私' | N2 == '私')  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords) %>%
    filter(str_detect(POS2, '非自立')) %>%
    arrange(desc(Row3)) %>% head(10)

In [ ]:
bigram %>% 
    filter(N1 == '私' | N2 == '私')  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords) %>%
    filter(!str_detect(POS2, '非自立')) %>%
    arrange(desc(Row3)) %>% head(10)

In [ ]:
bigram %>% 
    filter(N1 == '私' | N2 == '私')  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords) %>%
    filter(!str_detect(POS2, '非自立')) %>%
    filter(str_detect(POS2, '動詞')) %>%
    select(everything(), freq=Row3) %>% 
    arrange(desc(freq)) %>% head(10)


In [ ]:
bigram %>% 
    filter(N1 == '私' | N2 == '私')  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords) %>%
    filter(!str_detect(POS2, '非自立')) %>%
    filter(str_detect(POS1, '動詞')) %>%
    select(everything(), freq=Row3) %>% 
    arrange(desc(freq)) %>% head(10)

In [ ]:
net = bigram %>% 
    filter(N1 == '私' | N2 == '私')  %>% 
    filter(str_detect(POS1, '名詞-名詞')) %>%
    select(N1, N2, freq=Row3) %>% 
    filter(freq > 4)  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords)

net %>% graph_plot()

In [ ]:
net = bigram %>%
    filter(N1 == '私' | N2 == '私') %>% 
    filter(str_detect(POS1, '動詞')) %>%
    select(N1, N2, freq=Row3) %>% 
    filter(freq > 4)  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords)

net %>% graph_plot()

In [ ]:
# 登場人物と形容詞の結びつき
net = bigram %>% 
    filter(N1 %in% c('私','Ｋ','お嬢さん') | N2 %in% c('私','Ｋ','お嬢さん')) %>% 
    filter(str_detect(POS1, '形容')) %>%
    select(N1, N2, freq=Row3) %>% 
    filter(freq > 0)  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords)

net %>% graph_plot()

In [ ]:
### Bag of wordsを用いた共起関係の分析
## sentenceごとに共起を集計するため、sentenceごとに分けたデータを読み込む

kokoro_sentence = read.delim('data/textmining/kokoro_sentence.tsv', header=T, sep='\t', stringsAsFactor=F, fileEncoding='utf8')
kokoro_sentence %>% head()

In [ ]:
# 定義済みの関数を読み込む
source('functions.r', encoding='utf8') 

stopwords = c('事','の','よう','それ','もの', '人', '何','一', 'ん','方','二','前','気','中','上','今','ため', '時', 'そこ', 'どこ', 'これ', 'そう',
              'いる', 'なる', 'する', 'いう', 'ある', 'れる', 'られる', 'くれる')

# map(): ベクトルの各要素に関数を適用する
res = map(kokoro_sentence[,'content'], get_cooc, pos=c('名詞'), stopwords=stopwords)  %>% unlist() %>% table() # 共起を集計する


In [ ]:
options(repr.plot.width=6, repr.plot.height=6)

In [ ]:
# データ構造と分布
res %>% dim()
res %>% as.vector() %>% sort(decreasing=T) %>% plot()
res %>% as.vector() %>% sort(decreasing=T) %>% plot(log='y') # 片対数
res %>% as.vector() %>% sort(decreasing=T) %>% plot(log='xy')



In [ ]:
# 各部ごとに共起を集計する
part1 = map(kokoro_sentence[kokoro_sentence$part_id == 1,'content'], get_cooc, pos=c('名詞'), stopwords=stopwords) %>% unlist() %>% table()
part2 = map(kokoro_sentence[kokoro_sentence$part_id == 2,'content'], get_cooc, pos=c('名詞'), stopwords=stopwords) %>% unlist() %>% table()
part3 = map(kokoro_sentence[kokoro_sentence$part_id == 3,'content'], get_cooc, pos=c('名詞'), stopwords=stopwords) %>% unlist() %>% table()
part3 %>% head()

In [ ]:
df1 = as.data.frame(part1)
df2 = as.data.frame(part2)
df3 = as.data.frame(part3)

# １つのdataframeに統合する
res = merge(x=df1, y=df2, by='.', all=T)
res = merge(x=res, y=df3, by='.', all=T)
colnames(res) = c('term', 'df1', 'df2', 'df3')
res[is.na(res)] = 0 # 欠損を埋める

res %>% head()

In [ ]:
### 主成分分析

mat = res
row.names(mat) = mat[,1]
mat = mat[2:4]
mat = mat[rowSums(mat) > 50, ]
colnames(mat) = c('第一部','第二部','第三部')

mat_t = t(mat)

ratio = mat_t / colSums(mat)
ratio_t = t(ratio)
result = (ratio_t / colSums(ratio))  %>% prcomp()

biplot(result)


In [ ]:
stopwords = c('事','の','よう','それ','もの', '人', '何','一', 'ん','方','二','前','気','中','上','今','ため', '時', 'そこ', 'どこ', 'これ', 'そう',
              'いる', 'なる', 'する', 'いう', 'ある', 'れる', 'られる', 'くれる')


In [ ]:
## 名詞と形容詞の結びつき
res = map(kokoro_sentence[kokoro_sentence$part_id==3,'content'], get_cooc, pos=c('名詞', '形容詞'), with_pos=T, stopwords=stopwords, dic=dict)  %>%
    unlist() %>% table() 
res %>% head()


In [ ]:
# 描画用データに変換する
# parse_cooc > function.rを参照
df = parse_cooc(names(res), as.vector(res))


In [ ]:
# データの形状
df %>% head()
df$freq %>% sort(decreasing=T) %>% plot()
df$freq %>% sort(decreasing=T) %>% plot(log='y')


In [ ]:
# ネットワーク図　
net = df %>% 
    filter(N1 %in% c('私','Ｋ','お嬢さん') | N2 %in% c('Ｋ','お嬢さん','私')) %>% # どちらかの単語がK, お嬢さん, 私であるものを抽出
    filter(str_detect(POS, '形容')) %>% # 品詞に形容詞・形容動詞をを利用
    filter(freq > 0)  %>% 
    filter(! N1 %in% stopwords) %>%  
    filter(! N2 %in% stopwords)


In [ ]:
options(repr.plot.width=16, repr.plot.height=9)
net %>% graph_plot()